# Notebook 3 — Advanced Prompting Strategies

**Topics covered in this notebook:**
7. Prompt Chaining and Decomposition Strategies
8. Meta-Prompting and Self-Refinement Loops

---

## ⚙️ Setup — Pick your API provider (free options available!)

| Provider | Cost | Where to get a key |
|----------|------|--------------------|
| **Groq** ✅ FREE | No credit card | [console.groq.com/keys](https://console.groq.com/keys) |
| **Gemini** ✅ FREE | No credit card | [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) |
| **OpenAI** | Paid | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) |

> These techniques power real production AI systems — pipelines, agents, and automated feedback loops.
> By the end you'll be able to build multi-step reasoning systems from scratch.

In [ ]:
# ── CELL 0 · Install dependencies (run once) ─────────────────────────────────
import subprocess
result = subprocess.run(
    ["uv", "pip", "install",
     "openai>=3.8.0",
     "pydantic>=2.12.0",
     "python-dotenv>=1.0.0",
     "rich>=14.0.0"],
    capture_output=True, text=True
)
print(result.stdout or "All packages already installed.")

In [1]:
# ── CELL 1 · Provider auto-detection & client setup ─────────────────────────
import os
import json
from typing import Optional
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from rich import print as rprint
from rich.panel import Panel
from rich.rule import Rule
from rich.text import Text

load_dotenv()

# ── Provider detection — priority: OpenAI > Groq > Gemini ────────────────────
OPENAI_KEY  = os.getenv("OPENAI_API_KEY", "")
GROQ_KEY    = os.getenv("GROQ_API_KEY", "")
GEMINI_KEY  = os.getenv("GEMINI_API_KEY", "")

if OPENAI_KEY and not OPENAI_KEY.startswith("sk-..."):
    PROVIDER = "openai"
    client = OpenAI(api_key=OPENAI_KEY)
    MODEL = "gpt-4o"
elif GROQ_KEY and not GROQ_KEY.startswith("gsk_..."):
    PROVIDER = "groq"
    client = OpenAI(
        api_key=GROQ_KEY,
        base_url="https://api.groq.com/openai/v1",
    )
    MODEL = "openai/gpt-oss-120b"
elif GEMINI_KEY and not GEMINI_KEY.startswith("AIza..."):
    PROVIDER = "gemini"
    client = OpenAI(
        api_key=GEMINI_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    MODEL = "models/gemini-3.7-flash"
else:
    raise EnvironmentError(
        "No valid API key found in .env!\n"
        "Add one of: OPENAI_API_KEY, GROQ_API_KEY, or GEMINI_API_KEY\n"
        "See .env.example for instructions."
    )

print(f"✓ Provider : {PROVIDER.upper()}")
print(f"✓ Model    : {MODEL}\n")

# ── Shared helper functions ───────────────────────────────────────────────────
def chat(messages: list[dict], model: str = MODEL, temperature: float = 0.3,
         response_format: dict | None = None) -> str:
    kwargs = dict(model=model, messages=messages, temperature=temperature)
    if response_format:
        kwargs["response_format"] = response_format
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

def show(content: str, title: str, border: str = "green") -> None:
    rprint(Panel(str(content), title=f"[bold]{title}[/]", border_style=border))

def step_banner(n: int, title: str) -> None:
    rprint(Rule(f"[bold cyan]Step {n}: {title}[/]"))

✓ Provider : GROQ
✓ Model    : openai/gpt-oss-120b



---
## Part 7 — Prompt Chaining and Decomposition Strategies

**Prompt chaining** = breaking a complex task into a sequence of smaller, simpler prompts  
where the **output of one prompt becomes the input to the next**.

### Why chain instead of one mega-prompt?
- Each step can have a **specialized instruction** tailored to that sub-task
- You can **inspect and validate** each intermediate result
- Errors stay contained — easier to debug which step failed
- Allows **conditional branching** (route to different chains based on step output)
- Enables **parallel execution** of independent sub-tasks

### Chain patterns
| Pattern | Description | Use when |
|---------|------------|----------|
| **Sequential** | A → B → C (linear pipeline) | Ordered transformation |
| **Map-Reduce** | Process N items → aggregate | Summarizing many documents |
| **Parallel** | A + B (independent, then merge) | Faster multi-aspect analysis |
| **Gated/Conditional** | Route based on step output | Classification → specialized handler |

### Research insight
> Prompt chaining improves accuracy by **+10–30%** on complex multi-step tasks vs a single mega-prompt. *(SurePrompts, 2026)*  
> Each individual prompt in a chain should stay focused on **one clear task**.

In [2]:
# ── EXAMPLE 7a · Sequential chain — Document Analysis Pipeline ───────────────
# Pipeline: raw document → extract facts → verify facts → generate report

raw_document = """\
INTERNAL MEMO — Q3 2024 Performance Review

Revenue came in at $42.3M, up 18% YoY. However, CAC (Customer Acquisition Cost) 
increased 34% to $1,847 per customer. Churn rose to 6.2% from 4.1% last quarter. 
The new enterprise tier launched in July contributed $8.1M — ahead of the $6M forecast.
Engineering headcount grew from 47 to 63 (34% increase), driving OpEx up 28%.
Net margin compressed from 22% to 14% despite top-line growth.
"""

# ── Step 1: Extract key metrics ─────────────────────────────────────────────
step_banner(1, "Extract Key Metrics")
step1_prompt = f"""\
Extract all numerical metrics from this business memo.
Return a JSON object where each key is a metric name (snake_case) 
and each value is the number (as a float or percentage string).
Include YoY/QoQ comparisons as separate keys with suffix _prev.

Memo:
{raw_document}
"""
metrics_raw = chat([{"role": "user", "content": step1_prompt}],
                   response_format={"type": "json_object"}, temperature=0.0)
metrics = json.loads(metrics_raw)
rprint(metrics)

# ── Step 2: Identify concerning trends ──────────────────────────────────────
step_banner(2, "Identify Concerning Trends")
step2_prompt = f"""\
You are a CFO analyzing business metrics. Given these Q3 metrics:
{json.dumps(metrics, indent=2)}

Identify the top 3 most concerning trends that leadership should address urgently.
For each concern:
- Name the metric
- Explain WHY it's concerning (business impact)
- Suggest ONE specific investigation to run

Format: numbered list, concise.
"""
concerns = chat([{"role": "user", "content": step2_prompt}], temperature=0.2)
show(concerns, "Top Concerns Identified", "yellow")

# ── Step 3: Generate executive summary ──────────────────────────────────────
step_banner(3, "Generate Executive Summary")
step3_prompt = f"""\
Write a concise executive summary for the board.

Raw data:
{raw_document}

Key concerns flagged by CFO analysis:
{concerns}

Format:
- Headline (1 sentence with the #1 story)
- Positives (2 bullets)
- Risks (2 bullets, from the concerns above)
- Recommended board action (1 sentence)

Tone: Board-level, direct, no jargon.
"""
summary = chat([{"role": "user", "content": step3_prompt}], temperature=0.2)
show(summary, "Final Executive Summary (3-step chain)", "green")

─────────────────────────────────────────── Step 1: Extract Key Metrics ───────────────────────────────────────────

{
    'revenue': 42.3,
    'revenue_yoy': '18%',
    'cac': 1847,
    'cac_yoy': '34%',
    'churn': '6.2%',
    'churn_prev': '4.1%',
    'enterprise_tier_revenue': 8.1,
    'enterprise_tier_forecast': 6.0,
    'engineering_headcount': 63,
    'engineering_headcount_prev': 47,
    'engineering_headcount_yoy': '34%',
    'opex_yoy': '28%',
    'net_margin': '14%',
    'net_margin_prev': '22%'
}

─────────────────────────────────────── Step 2: Identify Concerning Trends ────────────────────────────────────────

╭──────────────────────────────────────────── Top Concerns Identified ────────────────────────────────────────────╮
│ 1. **Customer Churn (6.2% ↑ from 4.1%)**                                                                        │
│    - **Why it’s concerning:** A 2.1‑point jump signals accelerating loss of revenue and higher replacement      │
│ costs, threatening growth and lifetime value.                                                                   │
│    - **Investigation:** Run a cohort‑level churn analysis (by product tier, acquisition channel, and contract   │
│ length) to pinpoint which segments are defecting and why.                                                       │
│                                                                                                                 │
│ 2. **Customer Acquisition Cost (CAC $1,847 ↑ 34% YoY)**                                                         │
│    - **Why it’s concerning:** A 34 % rise inflates the pay‑back period and erodes unit economics, especially    │
│ when paired with higher churn. It may indicate inefficient marketing spend or a shift to higher‑cost channels.  │
│    - **Investigation:** Conduct a CAC‑by‑channel attribution study to compare cost, conversion rates, and LTV   │
│ across all acquisition sources and identify low‑performing spend.                                               │
│                                                                                                                 │
│ 3. **Net Margin (14% ↓ from 22%)**                                                                              │
│    - **Why it’s concerning:** An 8‑point margin compression reflects weaker profitability despite revenue       │
│ growth, likely driven by rising OPEX (28 % YoY) and the higher CAC. Sustained margin pressure threatens cash    │
│ flow and valuation.                                                                                             │
│    - **Investigation:** Perform a detailed OPEX variance analysis (break down by R&D, sales & marketing, G&A)   │
│ to isolate cost drivers and assess opportunities for efficiency or re‑prioritization.                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────── Step 3: Generate Executive Summary ────────────────────────────────────────

╭──────────────────────────────────── Final Executive Summary (3-step chain) ─────────────────────────────────────╮
│ **Headline:** Q3 revenue surged 18% to $42.3 M, but margin fell 8 points as churn and acquisition costs         │
│ accelerated.                                                                                                    │
│                                                                                                                 │
│ **Positives**                                                                                                   │
│ - Enterprise tier launch delivered $8.1 M, beating the $6 M forecast.                                           │
│ - Top‑line growth outpaced the market, confirming demand for our new product tier.                              │
│                                                                                                                 │
│ **Risks**                                                                                                       │
│ - Customer churn jumped to 6.2% (up 2.1 pts), eroding recurring revenue and raising replacement costs.          │
│ - CAC rose 34% to $1,847, lengthening pay‑back and squeezing unit economics.                                    │
│                                                                                                                 │
│ **Recommended board action:** Approve a focused cost‑efficiency task force to audit acquisition spend and churn │
│ drivers, and to prioritize margin‑protective initiatives before the next quarter.                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
# ── EXAMPLE 7b · Map-Reduce chain — Summarize multiple documents ─────────────
# Map: summarize each document independently
# Reduce: synthesize all summaries into one final output

# Simulating 4 customer reviews to aggregate
reviews = [
    """Review 1: Setup took over an hour — the instructions are terrible. 
    Once running though, it's fast and reliable. WiFi range is impressive.""",
    
    """Review 2: Plug and play, worked in 10 minutes. The app is intuitive. 
    However, it disconnects every few days and needs a manual restart.""",
    
    """Review 3: Best router I've owned. Handles 30+ devices without slowdown. 
    The parental controls are excellent. Price is high but worth it.""",
    
    """Review 4: Returned it after a week. Customer support was unhelpful 
    when I had connectivity issues. Build quality feels cheap for the price.""",
]

# ── MAP phase: summarize each review independently ───────────────────────────
print("MAP phase — summarizing each review...")
map_template = """\
Summarize this product review in exactly 3 key points:
1. Main positive (if any)
2. Main negative (if any)  
3. Overall verdict in 5 words

Review: {review}
"""

summaries = []
for i, review in enumerate(reviews, 1):
    summary = chat([{"role": "user", "content": map_template.format(review=review)}], temperature=0.0)
    summaries.append(summary)
    print(f"  ✓ Review {i} summarized")

# ── REDUCE phase: synthesize all summaries ───────────────────────────────────
print("\nREDUCE phase — synthesizing all summaries...")
all_summaries = "\n\n".join(f"[Review {i+1}]\n{s}" for i, s in enumerate(summaries))

reduce_prompt = f"""\
You've received summaries of 4 customer reviews for a WiFi router.
Synthesize them into a balanced product assessment.

Individual summaries:
{all_summaries}

Write:
1. Overall rating (1–5 stars) with one-sentence justification
2. Top 3 strengths (mentioned by multiple reviewers)
3. Top 3 weaknesses (mentioned by multiple reviewers)
4. Who should buy this? (1 sentence)
5. Who should avoid this? (1 sentence)
"""

final_assessment = chat([{"role": "user", "content": reduce_prompt}], temperature=0.2)
show(final_assessment, "Map-Reduce: Aggregated Product Assessment", "green")

MAP phase — summarizing each review...
  ✓ Review 1 summarized
  ✓ Review 2 summarized
  ✓ Review 3 summarized
  ✓ Review 4 summarized

REDUCE phase — synthesizing all summaries...


╭─────────────────────────────────── Map-Reduce: Aggregated Product Assessment ───────────────────────────────────╮
│ **Overall rating:** ★★★☆☆ – The router delivers fast, wide‑area Wi‑Fi and can juggle many devices, but its      │
│ cumbersome setup, intermittent drops and pricey, flimsy build keep it from being a truly reliable choice.       │
│                                                                                                                 │
│ **Top 3 strengths (cited by more than one reviewer)**                                                           │
│ 1. **Strong Wi‑Fi performance & coverage** – reviewers praise its speed, reliability and impressive range.      │
│ 2. **Capacity for many devices** – it handles 30 + simultaneous connections without slowing down.               │
│ 3. **Convenient management app** – the plug‑and‑play app is noted as intuitive and useful for settings like     │
│ parental controls.                                                                                              │
│                                                                                                                 │
│ **Top 3 weaknesses (cited by more than one reviewer)**                                                          │
│ 1. **Unreliable connectivity** – frequent disconnections and overall connectivity problems are reported.        │
│ 2. **Poor onboarding & support** – confusing or missing setup instructions and unhelpful customer service       │
│ frustrate users.                                                                                                │
│ 3. **High price vs. build quality** – the premium cost feels unjustified given the cheap‑feel hardware and      │
│ durability concerns.                                                                                            │
│                                                                                                                 │
│ **Who should buy this?**                                                                                        │
│ Anyone who needs robust Wi‑Fi coverage for many devices and values a feature‑rich app for parental controls and │
│ network management.                                                                                             │
│                                                                                                                 │
│ **Who should avoid this?**                                                                                      │
│ Users who prioritize hassle‑free installation, rock‑solid reliability, or a budget‑friendly, well‑built router. │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
# ── EXAMPLE 7c · Gated/Conditional chain — Support Ticket Router ─────────────
# Step 1: Classify ticket category
# Step 2: Route to the appropriate specialized handler based on category
# This is how real support automation pipelines work.

# Specialized response handlers for each category
HANDLERS = {
    "Billing": """\
You are a billing specialist. Be empathetic and precise.
Always mention: our billing cycle, how to download invoices, and escalation path to finance team.
Offer a specific resolution, not generic "we'll look into it".""",

    "Technical Bug": """\
You are a Level 2 technical support engineer. Be systematic.
Always ask for: browser/OS version, steps to reproduce, error messages.
Provide immediate workarounds if available, then log the bug for engineering.""",

    "Feature Request": """\
You are a product manager. Be appreciative and transparent.
Acknowledge the request, explain our prioritization process, and ask for use case details.
Never promise a timeline. Do add to the feedback database.""",

    "Account Access": """\
You are a security-focused account specialist.
Prioritize account security above all. Follow our verification protocol before any changes.
Provide self-service options (password reset link, 2FA setup) before manual intervention.""",
}

def route_and_respond(ticket: str) -> tuple[str, str]:
    """Classify a support ticket then generate a specialized response."""
    
    # Gate: classify first
    classify_prompt = f"""\
Classify this support ticket into exactly one category.
Choose from: Billing, Technical Bug, Feature Request, Account Access, Other
Return only the category name, nothing else.

Ticket: "{ticket}"
Category:"""
    category = chat([{"role": "user", "content": classify_prompt}], temperature=0.0).strip()
    
    # Route to specialized handler (or fallback to generic)
    system_prompt = HANDLERS.get(category, "You are a helpful customer support agent.")
    
    response = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Support ticket: {ticket}"}
    ], temperature=0.3)
    
    return category, response


# Test with 3 different tickets
tickets = [
    "I was charged twice last month and the charge appeared on my card but not in my dashboard.",
    "The CSV export button does nothing when I click it in Chrome. I'm on macOS 14.",
    "Would be amazing if you added dark mode! It would make night-time usage much easier.",
]

for ticket in tickets:
    category, response = route_and_respond(ticket)
    rprint(Rule(f"[bold]Ticket:[/] {ticket[:60]}..."))
    rprint(f"[bold yellow]→ Routed to:[/] {category}")
    show(response, f"{category} Handler Response", "blue")
    print()

───────────────────── Ticket: I was charged twice last month and the charge appeared on my... ─────────────────────

→ Routed to: Billing

╭─────────────────────────────────────────── Billing Handler Response ────────────────────────────────────────────╮
│ **Hi [Your Name],**                                                                                             │
│                                                                                                                 │
│ I’m really sorry to see that you’ve been charged twice for last month’s subscription and that the second charge │
│ isn’t even showing up in your dashboard. I understand how frustrating that can be, and I’m here to get this     │
│ sorted for you quickly.                                                                                         │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### What’s happening?                                                                                           │
│                                                                                                                 │
│ - **Billing cycle:** Our service runs on a **monthly billing cycle** that renews on the same calendar day each  │
│ month (e.g., the 15th → 15th).                                                                                  │
│ - **Duplicate charge:** A duplicate charge usually occurs when the payment processor receives two authorization │
│ attempts for the same cycle. Because the second attempt never completed in our system, it doesn’t appear on the │
│ dashboard, but the card issuer still posted the amount.                                                         │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### Immediate next steps (specific resolution)                                                                  │
│                                                                                                                 │
│ 1. **Verify the duplicate transaction**                                                                         │
│    Please reply with the following details so I can pinpoint the exact charge:                                  │
│    - The **date** the duplicate charge posted on your card statement.                                           │
│    - The **last four digits** of the card used.                                                                 │
│    - The **amount** of each charge (they should be identical).                                                  │
│                                                                                                                 │
│ 2. **Refund / credit processing**                                                                               │
│    Once I have those details, I will:                                                                           │
│    - **Submit a refund request** to our payment processor for the duplicate amount. Refunds typically appear on │
│ your card within **5‑7 business days**.                                                                         │
│    - **Apply a credit** to your account for the current billing cycle, ensuring you won’t be out‑of‑pocket      │
│ while the refund is in transit.                                                                                 │
│                                                                                                                 │
│ 3. **Confirm on the dashboard**                       

───────────────────── Ticket: The CSV export button does nothing when I click it in Chrome... ─────────────────────

→ Routed to: Technical Bug

╭──────────────────────────────────────── Technical Bug Handler Response ─────────────────────────────────────────╮
│ **Thank you for reaching out!** I’m sorry you’re having trouble with the CSV export button. Let’s gather a few  │
│ more details so I can pinpoint the cause and get you a fix as quickly as possible.                              │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. Environment Details                                                                                      │
│ | Item | What I need from you |                                                                                 │
│ |------|----------------------|                                                                                 │
│ | **Browser** | Exact Chrome version (e.g., Chrome 118.0.5993.90). You can find this under **Chrome > About     │
│ Google Chrome**. |                                                                                              │
│ | **Operating System** | You mentioned macOS 14 – could you confirm the exact build number (e.g., **macOS       │
│ 14.2.1**) from **Apple > About This Mac**? |                                                                    │
│ | **Any extensions or ad‑blockers** | List any Chrome extensions you have enabled, especially privacy,          │
│ ad‑blocking, or script‑blocking ones. |                                                                         │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 2. Steps to Reproduce                                                                                       │
│ Please let me know the exact sequence you follow, for example:                                                  │
│                                                                                                                 │
│ 1. Log in → navigate to **Reports → Sales** → click **Export CSV**.                                             │
│ 2. …or any other page where the button appears.                                                                 │
│                                                                                                                 │
│ If there are any variations (different pages, filters applied, etc.), include those as well.                    │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 3. Error Messages / Console Output                                                                          │
│ - **On‑screen messages:** Does anything appear at the top or bottom of the screen after you click the button    │
│ (e.g., “Something went wrong” or a silent toast that disappears quickly)?                                       │
│ - **Browser console:** Open Chrome DevTools (**⌥ ⌘ I**), go to the **Console** tab, click the CSV button again, │
│ and copy any red error lines that appear.                                                                       │
│   *(If you’re not comfortable with the console, let me know and I can walk you through it.)*                    │
│                                                       

───────────────────── Ticket: Would be amazing if you added dark mode! It would make night... ─────────────────────

→ Routed to: Feature Request

╭─────────────────────────────────────── Feature Request Handler Response ────────────────────────────────────────╮
│ Thank you for taking the time to share this idea with us! 🎉                                                    │
│ We completely understand how valuable a dark‑mode option would be for night‑time usage, and we appreciate you   │
│ bringing it to our attention.                                                                                   │
│                                                                                                                 │
│ **How we prioritize new features**                                                                              │
│ When we evaluate feature requests, we look at a combination of factors: the size and impact of the user need,   │
│ alignment with our product strategy, technical feasibility, and the overall benefit to the broader user         │
│ community. Each request is logged, reviewed regularly by the product team, and weighed against other            │
│ initiatives that are already in progress.                                                                       │
│                                                                                                                 │
│ **Next steps**                                                                                                  │
│ To help us assess the potential impact of dark mode more accurately, could you share a bit more about how you’d │
│ use it? For example:                                                                                            │
│                                                                                                                 │
│ - Which parts of the product would you most like to see in dark mode (e.g., dashboards, reports, settings       │
│ pages)?                                                                                                         │
│ - How often do you work during evening or low‑light conditions?                                                 │
│ - Are there any accessibility or visual‑comfort considerations that are especially important for you?           │
│                                                                                                                 │
│ Your additional context will be extremely helpful as we evaluate this request alongside others.                 │
│                                                                                                                 │
│ Rest assured, we’ve added your suggestion to our feedback database, and it will be reviewed during our upcoming │
│ prioritization cycle. While I can’t commit to a specific timeline, please know that we’ll keep you posted on    │
│ any developments.                                                                                               │
│                                                                                                                 │
│ Thanks again for helping us make the product better! If you have any other ideas or details to share, just let  │
│ me know.                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
# ── EXAMPLE 7d · Parallel decomposition — Code review pipeline ───────────────
# Run multiple specialized reviewers in parallel (independent → merge)
# In production this would use asyncio; here we show the pattern synchronously.

code_to_review = """\
import requests

def get_user_data(user_id):
    url = f"http://api.internal.com/users/{user_id}"
    response = requests.get(url, timeout=30)
    data = response.json()
    
    # Cache in memory for performance
    cache = {}
    cache[user_id] = data
    
    password = data.get('password_hash')
    print(f"Retrieved user {user_id}: password={password}")
    
    return data

def process_users(user_ids):
    results = []
    for uid in user_ids:
        result = get_user_data(uid)
        results.append(result)
    return results
"""

# Three specialized reviewers — run independently, then merge
reviewers = {
    "Security": "You are a cybersecurity expert. Review ONLY for security vulnerabilities, data leaks, and unsafe patterns. Ignore style.",
    "Performance": "You are a performance engineer. Review ONLY for inefficiencies, unnecessary operations, and scalability issues. Ignore security.",
    "Code Quality": "You are a senior engineer focused on code quality. Review ONLY for maintainability, error handling, and best practices. Ignore performance.",
}

review_results = {}
for role, system in reviewers.items():
    review_prompt = f"Review this Python code:\n```python\n{code_to_review}\n```\nProvide 3 specific findings with severity (Critical/High/Medium/Low) and a one-line fix recommendation each."
    review_results[role] = chat([
        {"role": "system", "content": system},
        {"role": "user", "content": review_prompt}
    ], temperature=0.1)

# Merge: synthesize all reviews into a prioritized action list
all_reviews = "\n\n".join(f"=== {role} Review ===\n{review}" for role, review in review_results.items())
merge_prompt = f"""\
You received three independent code reviews (security, performance, quality).
Merge them into a single prioritized action list.

Rules:
- Eliminate duplicates
- Sort by priority: Critical > High > Medium > Low
- Format each item: [PRIORITY] Description — Fix in one line
- Add a one-line "Ship it?" verdict at the top

Reviews:
{all_reviews}
"""

merged = chat([{"role": "user", "content": merge_prompt}], temperature=0.1)

for role, review in review_results.items():
    show(review, f"{role} Reviewer", "yellow")
print()
show(merged, "Merged & Prioritized Code Review (Parallel Chain)", "green")

╭─────────────────────────────────────────────── Security Reviewer ───────────────────────────────────────────────╮
│ **Finding 1 – Insecure transport (HTTP)**                                                                       │
│ - **Severity:** High                                                                                            │
│ - **Issue:** The code calls the internal API over plain‑text `http://`, exposing the request (including the     │
│ user ID) to network eavesdropping and man‑in‑the‑middle attacks.                                                │
│ - **Fix:** Switch to `https://` and ensure TLS certificate verification (the default in `requests`) is enabled. │
│                                                                                                                 │
│ **Finding 2 – Sensitive data leakage (printing password hash)**                                                 │
│ - **Severity:** Critical                                                                                        │
│ - **Issue:** `print(f"Retrieved user {user_id}: password={password}")` writes the password hash to stdout/logs, │
│ potentially exposing credential material to anyone with access to logs or the console.                          │
│ - **Fix:** Remove the password hash from any logging/printing and, if logging is needed, mask or omit it        │
│ entirely.                                                                                                       │
│                                                                                                                 │
│ **Finding 3 – Potential SSRF / unsanitized input in URL construction**                                          │
│ - **Severity:** Medium                                                                                          │
│ - **Issue:** The `user_id` value is interpolated directly into the URL without validation, allowing an attacker │
│ to supply malicious strings (e.g., `../../etc/passwd`) that could cause the request to reach unintended         │
│ internal endpoints.                                                                                             │
│ - **Fix:** Validate and whitelist `user_id` (e.g., ensure it is an integer or matches a strict regex) before    │
│ embedding it in the request URL.                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Performance Reviewer ──────────────────────────────────────────────╮
│ **Finding 1 – Cache recreated on every call (Low)**                                                             │
│ *Issue:* `cache = {}` is instantiated inside `get_user_data`, so the dictionary is discarded after the function │
│ returns; the intended in‑memory caching never works.                                                            │
│ *Fix:* Define the cache once at module level (or use `functools.lru_cache`) and store/retrieve values from that │
│ shared object.                                                                                                  │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ **Finding 2 – Sequential blocking HTTP requests (High)**                                                        │
│ *Issue:* `process_users` loops over `user_ids` and calls `requests.get` synchronously, causing the total        │
│ latency to be the sum of all round‑trips; this does not scale beyond a few dozen users.                         │
│ *Fix:* Switch to an asynchronous HTTP client (e.g., `httpx.AsyncClient` or `aiohttp`) or a thread/worker pool   │
│ to fetch many users in parallel.                                                                                │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ **Finding 3 – Unnecessary per‑call I/O (Medium)**                                                               │
│ *Issue:* `print(f"Retrieved user {user_id}: password={password}")` writes to stdout for every user, adding      │
│ costly I/O and potentially leaking sensitive data in logs.                                                      │
│ *Fix:* Remove the print statement or replace it with a conditional debug‑log call (`logger.debug(...)`) that    │
│ can be disabled in production.                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Code Quality Reviewer ─────────────────────────────────────────────╮
│ **Finding 1 – Missing error handling for the HTTP request**                                                     │
│ - **Severity:** High                                                                                            │
│ - **Fix:** Wrap the `requests.get` call in a `try/except` block, call `response.raise_for_status()`, and catch  │
│ `requests.exceptions.RequestException` and `json.JSONDecodeError` to return a graceful fallback or raise a      │
│ domain‑specific exception.                                                                                      │
│                                                                                                                 │
│ **Finding 2 – In‑function cache is recreated on every call and never reused**                                   │
│ - **Severity:** Medium                                                                                          │
│ - **Fix:** Move the `cache` dictionary to module scope (or use `functools.lru_cache`) so that cached data       │
│ persists across calls and can actually improve reuse.                                                           │
│                                                                                                                 │
│ **Finding 3 – Sensitive information (password hash) is printed to stdout**                                      │
│ - **Severity:** Critical                                                                                        │
│ - **Fix:** Remove the `print` statement (or at least mask the value) to avoid leaking credential data; use      │
│ proper logging with sanitized fields if debugging is needed.                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────── Merged & Prioritized Code Review (Parallel Chain) ───────────────────────────────╮
│ **Ship it?** No – resolve the critical and high‑severity issues before releasing.                               │
│                                                                                                                 │
│ [Critical] Sensitive data leakage – printing password hashes to stdout/logs — Remove the print statement (or    │
│ mask the hash) and use sanitized logging only.                                                                  │
│ [High] Insecure transport (HTTP) — Switch all internal API calls to `https://` and ensure TLS verification is   │
│ enabled.                                                                                                        │
│ [High] Sequential blocking HTTP requests — Replace the synchronous loop with an async client (e.g.,             │
│ `httpx.AsyncClient`) or a thread/worker pool to fetch users in parallel.                                        │
│ [High] Missing error handling for HTTP request — Wrap `requests.get` in `try/except`, call                      │
│ `response.raise_for_status()`, and handle `RequestException` and `JSONDecodeError` appropriately.               │
│ [Medium] Potential SSRF / unsanitized input in URL construction — Validate and whitelist `user_id` (e.g.,       │
│ enforce integer type or strict regex) before embedding it in the request URL.                                   │
│ [Medium] In‑function cache recreated on every call — Define the cache at module scope or apply                  │
│ `functools.lru_cache` so cached data persists across calls.                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 🧠 Student Exercise 7
Build a **3-step blog post pipeline**:
- **Step 1**: Given a topic (e.g., "The impact of AI on healthcare"), generate 5 different headline options
- **Step 2**: Score each headline on: clickability (1-10), clarity (1-10), SEO potential (1-10). Pick the best one.
- **Step 3**: Using the winning headline, generate a 400-word blog post outline (title + 5 sections with bullet points)

Print the output of each step with a clear separator between them.

---
## Part 8 — Meta-Prompting and Self-Refinement Loops

### What is meta-prompting?
**Meta-prompting** = using an LLM to **write, evaluate, or improve prompts** — rather than writing them by hand.

### Self-refinement loop pattern
```
1. GENERATE   → LLM produces an initial response
2. CRITIQUE   → LLM (or same LLM) evaluates the response against criteria
3. REFINE     → LLM revises the response based on the critique
4. (repeat N times or until quality threshold met)
```

### Key variants
| Variant | Description | Best for |
|---------|------------|----------|
| **Generate-Critique-Revise** | Single model, 3-step loop | Writing, code quality |
| **Self-Consistency** | Generate N responses, pick majority | Factual Q&A, math |
| **Meta-Prompt Generator** | Generate → evaluate prompts themselves | Prompt optimization |
| **Constitutional Refinement** | Critique against a set of principles | Safety-critical outputs |

### Research insight
> Self-refinement consistently improves quality by **10–25%** over single-pass generation. *(Lushbinary, 2026)*  
> Meta-prompting that writes prompts can match expert-written prompts on 70% of tasks. *(SurePrompts, 2026)*

In [6]:
# ── EXAMPLE 8a · Meta-Prompt Generator — LLM writes better prompts ───────────
# Give the model a weak task description → ask it to generate the optimal prompt

weak_description = "I want a prompt that makes the AI help me write marketing emails."

meta_prompt = f"""\
You are a prompt engineering expert. Your job is to take a vague task description 
and generate a high-quality, production-ready prompt for it.

A high-quality prompt includes:
1. Clear system role / persona
2. Specific task instruction (not vague)
3. Required input placeholders (use {{{{variable_name}}}} syntax)
4. Explicit output format with structure
5. 2-3 example outputs showing the expected style
6. Quality constraints (tone, length, what to avoid)

Task description from user:
"{weak_description}"

Generate the complete, production-ready prompt. Put it inside <prompt></prompt> tags.
After the tags, add a 3-bullet explanation of your design decisions.
"""

result = chat([{"role": "user", "content": meta_prompt}], temperature=0.4)
show(result, "Meta-Prompt Generator Output", "magenta")

# Extract and test the generated prompt
import re
generated_prompt = re.search(r'<prompt>(.*?)</prompt>', result, re.DOTALL)
if generated_prompt:
    print("\n✓ Prompt extracted. Testing it...")
    test_input = generated_prompt.group(1).strip()
    # Replace placeholders with actual values for a quick test
    test_prompt = test_input.replace("{product}", "CloudSync Pro").replace("{audience}", "small business owners")\
                            .replace("{goal}", "free trial signup").replace("{tone}", "professional but friendly")
    test_output = chat([{"role": "user", "content": test_prompt}], temperature=0.5)
    show(test_output[:800] + "..." if len(test_output) > 800 else test_output,
         "Testing the Generated Prompt", "green")

╭───────────────────────────────────────── Meta-Prompt Generator Output ──────────────────────────────────────────╮
│ <prompt>                                                                                                        │
│ **System Role**                                                                                                 │
│ You are a veteran marketing copywriter with 10+ years of experience crafting high‑conversion email campaigns    │
│ for B2B and B2C audiences. You understand persuasive psychology, brand voice consistency, and email             │
│ best‑practice guidelines (spam compliance, mobile‑friendly formatting, clear CTAs).                             │
│                                                                                                                 │
│ **Task**                                                                                                        │
│ Using the information provided in the placeholders, write a complete marketing email that persuades the target  │
│ audience to take the specified action. The email must include a subject line, a preheader, and a body that      │
│ follows the structure outlined below.                                                                           │
│                                                                                                                 │
│ **Required Input Placeholders**                                                                                 │
│ - `{{product_name}}` – name of the product or service.                                                          │
│ - `{{target_audience}}` – brief description of the ideal recipient (e.g., “small‑business owners in the tech    │
│ sector”).                                                                                                       │
│ - `{{email_goal}}` – the primary objective (e.g., “drive sign‑ups for a free trial”).                           │
│ - `{{key_benefits}}` – 2‑4 bullet‑point benefits, each ≤ 12 words.                                              │
│ - `{{call_to_action}}` – the exact CTA button text (e.g., “Start My Free Trial”).                               │
│ - `{{tone}}` – desired tone (e.g., “friendly & professional”, “energetic & bold”).                              │
│                                                                                                                 │
│ **Output Format**                                                                                               │
│ ```                                                                                                             │
│ Subject: {{subject_line}}          # ≤ 60 characters, title‑case, no all‑caps                                   │
│ Preheader: {{preheader}}          # ≤ 100 characters, complements subject                                       │
│                                                                                                                 │
│ Hi {{first_name_or_generic_greeting}},                                                                          │
│                                                                                                                 │
│ {{opening_paragraph}}            # 1‑2 sentences, hook related to {{target_audience}}                           │
│                                                                                                                 │
│ {{body_paragraph}}                # 2‑3 sentences, weave in {{key_benefits}} with vivid language                │
│                                                                                                                 │
│ {{cta_paragraph}}                 # 1 sentence, introduce the CTA button                                        │
│ [{{call_to_action}}]              # display as a button label (plain text)                                      │
│                                                       


✓ Prompt extracted. Testing it...


╭───────────────────────────────────────── Testing the Generated Prompt ──────────────────────────────────────────╮
│ Subject: Discover How {{product_name}} Empowers {{target_audience}}                                             │
│ Preheader: Unlock {{key_benefits}} and boost results with {{product_name}} today.                               │
│                                                                                                                 │
│ Hi {{first_name_or_generic_greeting}},                                                                          │
│                                                                                                                 │
│ Running a {{target_audience}} means balancing priorities while staying ahead of the competition.                │
│                                                                                                                 │
│ {{product_name}} gives you the edge you need. With {{key_benefits}} you’ll enjoy smoother operations, faster    │
│ decisions, and measurable growth—all without extra hassle.                                                      │
│                                                                                                                 │
│ Imagine a solution that works as hard as you do, delivering results that speak for themselves. That’s the       │
│ promise of {{product_name}}, and it’s ready to transform the way you work.                                      │
│                                                                                                                 │
│ Ready to experience the difference? Click the button below to {{email_goal}}.                                   │
│                                                                                                                 │
│ [{{call_to_action}}]                                                                                            │
│                                                                                                                 │
│ Best regards,                                                                                                   │
│ {{your_name}}                                                                                                   │
│ {{your_title}}                                                                                                  │
│ {{comp...                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# ── EXAMPLE 8b · Generate-Critique-Revise Loop — Essay quality improvement ───
# This is the most common self-refinement pattern.
# Each iteration the essay gets better. We'll run 2 refinement rounds.

essay_topic = "The role of curiosity in scientific discovery"

# Critique criteria — the model will evaluate against these
critique_criteria = """\
Evaluate the essay on these 5 criteria (score 1–10 each, then give specific improvement notes):
1. Thesis clarity — Is the main argument crystal clear in the opening paragraph?
2. Evidence quality — Are claims supported with specific examples, not vague generalizations?
3. Flow — Do paragraphs connect logically? Are transitions smooth?
4. Originality — Does it say anything surprising or non-obvious?
5. Conclusion impact — Does it leave the reader with something to think about?

For each criterion scoring below 7, provide ONE specific, actionable revision note.
Format: [Criterion]: Score/10 — Note (if score < 7)
"""

# Round 0: Generate initial draft
rprint(Rule("[bold]Round 0 — Initial Draft[/]"))
generate_prompt = f"Write a 300-word essay on: '{essay_topic}'. Focus on depth over breadth."
essay = chat([{"role": "user", "content": generate_prompt}], temperature=0.7)
show(essay, "Draft v0 (Initial)", "yellow")

# Refinement loop
for round_num in range(1, 3):  # 2 refinement rounds
    rprint(Rule(f"[bold]Round {round_num} — Critique & Revise[/]"))
    
    # Critique
    critique = chat([{
        "role": "user",
        "content": f"Read this essay and evaluate it:\n\n{essay}\n\n{critique_criteria}"
    }], temperature=0.1)
    show(critique, f"Critique (Round {round_num})", "red")
    
    # Revise
    essay = chat([{
        "role": "user",
        "content": f"""\
Here is an essay and specific critique feedback.
Rewrite the essay to address EVERY critique point below.
Keep the same length (~300 words). Do not just add filler.

Original essay:
{essay}

Critique to address:
{critique}

Revised essay:"""
    }], temperature=0.5)
    show(essay, f"Revised Essay v{round_num}", "green")

───────────────────────────────────────────── Round 0 — Initial Draft ─────────────────────────────────────────────

╭────────────────────────────────────────────── Draft v0 (Initial) ───────────────────────────────────────────────╮
│ Curiosity is the engine that drives scientific discovery, turning the mundane into the unknown. It is not       │
│ merely a fleeting interest but a disciplined insistence on asking why, how, and what if. When a researcher      │
│ feels a gap between observation and explanation, curiosity compels the formulation of a question that resists   │
│ easy answers. This mental tension fuels the design of experiments, the refinement of models, and the            │
│ willingness to tolerate failure.                                                                                │
│                                                                                                                 │
│ Consider the case of Michael Faraday, whose curiosity about the relationship between electricity and magnetism  │
│ led him to the discovery of electromagnetic induction. He did not set out with a predetermined hypothesis;      │
│ instead, he followed the puzzling behavior of a compass needle near a changing electric current. His relentless │
│ probing transformed a curious flicker into a principle that underpins modern power generation. Faraday’s work   │
│ illustrates how curiosity can redirect attention from accepted frameworks to subtle anomalies that hold deeper  │
│ significance.                                                                                                   │
│                                                                                                                 │
│ Curiosity also shapes the methodological rigor of science. It demands that data be interrogated rather than     │
│ accepted, prompting replication, control, and statistical scrutiny. When a result defies expectation, a curious │
│ mind resists the temptation to dismiss it as error; instead, it seeks hidden variables or new mechanisms. This  │
│ attitude has yielded breakthroughs such as the discovery of the cosmic microwave background, an accidental      │
│ signal that, because of curiosity, was recognized as relic radiation from the Big Bang.                         │
│                                                                                                                 │
│ In contemporary research, curiosity is institutionalized through grant programs that reward high‑risk,          │
│ high‑reward projects. Yet the personal quality remains paramount: scientists must nurture an inner wonder that  │
│ persists despite bureaucracy. Ultimately, curiosity is the crucible where observation, imagination, and rigor   │
│ fuse, producing the novel insights that expand humanity’s understanding of nature. By fostering an environment  │
│ where questions are valued above quick answers, we ensure that curiosity continues to ignite the transformative │
│ discoveries that define each new scientific era for future generations globally.                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────── Round 1 — Critique & Revise ───────────────────────────────────────────

╭────────────────────────────────────────────── Critique (Round 1) ───────────────────────────────────────────────╮
│ **Thesis clarity:** 8/10                                                                                        │
│ **Evidence quality:** 8/10                                                                                      │
│ **Flow:** 7/10                                                                                                  │
│ **Originality:** 6/10 — *Add a more unexpected or nuanced example (e.g., a modern “failed” experiment that      │
│ later sparked a breakthrough) or explore a less‑obvious angle on curiosity, such as its role in                 │
│ interdisciplinary cross‑pollination, to give the essay a fresher insight.*                                      │
│ **Conclusion impact:** 7/10                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Revised Essay v1 ────────────────────────────────────────────────╮
│ Curiosity is the catalyst that transforms ordinary observation into scientific revolution; it is the            │
│ disciplined insistence on asking “why,” “how,” and “what if,” even when answers are inconvenient. This essay    │
│ argues that curiosity fuels discovery not only by generating questions, but also by linking disparate fields    │
│ and by turning apparent failures into breakthroughs.                                                            │
│                                                                                                                 │
│ Michael Faraday’s 19th‑century experiments illustrate the classic pattern. He did not begin with a hypothesis   │
│ about induction; he followed the puzzling swing of a compass needle near a changing current, letting the        │
│ anomaly dictate his next step. The resulting principle of electromagnetic induction reshaped energy generation  │
│ and showed how curiosity can redirect attention from accepted frameworks to subtle, overlooked effects.         │
│                                                                                                                 │
│ A more recent, less‑obvious example comes from the 2015 BICEP2 collaboration. The team announced evidence of    │
│ primordial gravitational waves—an apparent triumph that quickly unraveled when interstellar dust was identified │
│ as the true source. Rather than discarding the result, the community’s curiosity spurred a cross‑disciplinary   │
│ audit involving astrophysicists, statisticians, and data‑science experts. The ensuing methodological            │
│ refinements sharpened measurements of the cosmic microwave background and deepened our understanding of         │
│ foreground contamination, turning a “failed” claim into a lasting improvement in observational cosmology.       │
│                                                                                                                 │
│ Curiosity also drives interdisciplinary cross‑pollination. Biologists borrowing statistical‑mechanics tools     │
│ from physics have uncovered universal scaling laws in ecosystems, while chemists applying machine‑learning      │
│ algorithms—originally developed for image recognition—now predict reaction outcomes with unprecedented speed.   │
│ In each case, the willingness to question disciplinary borders expands the toolbox of inquiry.                  │
│                                                                                                                 │
│ Institutional mechanisms such as high‑risk grant programs encourage this mindset, yet the personal habit of     │
│ sustained wonder remains decisive. By cultivating environments where questions outrank quick answers, we        │
│ safeguard the iterative loop of doubt, experiment, and revision that underlies every major advance. In doing    │
│ so, we ensure that curiosity continues to ignite the transformative discoveries that will define the scientific │
│ era for generations to come.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────── Round 2 — Critique & Revise ───────────────────────────────────────────

╭────────────────────────────────────────────── Critique (Round 2) ───────────────────────────────────────────────╮
│ Thesis clarity: 9/10                                                                                            │
│ Evidence quality: 8/10                                                                                          │
│ Flow: 7/10                                                                                                      │
│ Originality: 6/10 — Add a more unexpected or novel case study (e.g., a recent serendipitous breakthrough in     │
│ synthetic biology) to illustrate a surprising way curiosity reshapes a field.                                   │
│ Conclusion impact: 7/10                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Revised Essay v2 ────────────────────────────────────────────────╮
│ Curiosity is the engine that turns routine observation into scientific revolution; it compels us to ask “why,”  │
│ “how,” and “what if,” even when the answers threaten established comfort zones. This essay argues that          │
│ curiosity fuels discovery in three ways: it generates the questions that launch research, it bridges disparate  │
│ disciplines, and it converts apparent failures into lasting breakthroughs.                                      │
│                                                                                                                 │
│ Michael Faraday’s 19th‑century work exemplifies the first mode. He did not start with a theory of induction; he │
│ followed the inexplicable swing of a compass needle near a changing current, letting the anomaly dictate his    │
│ next experiment. The resulting law of electromagnetic induction reshaped energy generation and showed how a     │
│ simple, unanswered observation can overturn accepted frameworks.                                                │
│                                                                                                                 │
│ A second illustration comes from the 2015 BICEP2 claim of primordial gravitational waves. When interstellar     │
│ dust was later identified as the true source, the episode did not end in defeat. Curiosity drove a              │
│ cross‑disciplinary audit—astrophysicists, statisticians, and data‑science experts collaborated to refine        │
│ foreground‑subtraction techniques. The episode sharpened cosmic‑microwave‑background measurements and left the  │
│ field with more reliable tools than before.                                                                     │
│                                                                                                                 │
│ The most unexpected case arises from synthetic biology. In 2023, researchers exploring why a bacterium’s        │
│ riboswitch responded to an unrelated metabolite discovered a hidden RNA‑protein interaction that could be       │
│ repurposed as a programmable switch. Their curiosity about a “mistake” in gene‑expression data led to a modular │
│ RNA circuit that now directs cell‑type‑specific drug delivery, a breakthrough that was not predicted by any     │
│ existing design paradigm. The serendipitous insight illustrates how curiosity can reshape an entire field by    │
│ turning a puzzling outlier into a versatile technology.                                                         │
│                                                                                                                 │
│ Institutional incentives—high‑risk grant programs, open‑data mandates—nurture this mindset, but the personal    │
│ habit of sustained wonder remains decisive. By cultivating environments where questions outrank quick answers,  │
│ we preserve the iterative loop of doubt, experiment, and revision that underlies every major advance. In doing  │
│ so, we ensure that curiosity continues to ignite the transformative discoveries that will define the scientific │
│ era for generations to come.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [13]:
# ── EXAMPLE 8c · Self-Consistency Voting — for factual reliability ────────────
# Generate N independent answers, then pick the majority answer.
# Especially powerful for questions with one correct answer (math, factual recall).
# Research: +17.9% accuracy on GSM8K (Wang et al., 2022)

from collections import Counter

def self_consistency(question: str, n_samples: int = 5, temperature: float = 0.7) -> dict:
    """
    Generate n_samples independent answers and return the majority answer
    along with confidence (fraction of votes for winner).
    """
    answers = []
    for i in range(n_samples):
        prompt = f"""\
Think through this step by step, then give ONLY the final numerical answer 
(just the number, no units or explanation).

Question: {question}
Final answer:"""
        answer = chat([{"role": "user", "content": prompt}], temperature=temperature)
        # Extract just the number
        number = re.search(r'-?\d+\.?\d*', answer.strip())
        if number:
            answers.append(number.group())
        print(f"  Sample {i+1}: {answer.strip()[:30]}")
    
    vote_counts = Counter(answers)
    winner, count = vote_counts.most_common(1)[0]
    confidence = count / len(answers)
    return {"answer": winner, "confidence": confidence, "all_votes": dict(vote_counts)}


# Test on a problem where direct prompting often fails
problem = """\
A store offers 20% off, then an additional 15% off the discounted price. 
A customer also has a $10 coupon applied after both discounts.
Original price: $200. What is the final price the customer pays?
"""

print("Running self-consistency (5 independent samples)...")
result = self_consistency(problem, n_samples=5)

print(f"\n📊 Vote distribution: {result['all_votes']}")
print(f"✓ Majority answer: {result['answer']}")
print(f"📈 Confidence: {result['confidence']*100:.0f}%")

# The correct answer: $200 × 0.80 × 0.85 - $10 = $126
print(f"\n✅ Correct answer: $126 (200 × 0.80 × 0.85 − 10)")

Running self-consistency (5 independent samples)...
  Sample 1: 126
  Sample 2: 126
  Sample 3: 126
  Sample 4: 126
  Sample 5: 126

📊 Vote distribution: {'126': 5}
✓ Majority answer: 126
📈 Confidence: 100%

✅ Correct answer: $126 (200 × 0.80 × 0.85 − 10)


In [15]:
# ── EXAMPLE 8d · Prompt Optimization Loop — meta-prompting to improve prompts ─
# This is a full meta-prompting loop:
# 1. Start with a prompt
# 2. Test it on examples
# 3. Have the model critique the prompt (not the output)
# 4. Generate an improved prompt
# 5. Repeat

# The task: classify email urgency as High/Medium/Low
test_cases = [
    {"email": "Server is down, 500 errors, users can't log in!", "expected": "High"},
    {"email": "Can we schedule a call next week to discuss Q4 planning?", "expected": "Low"},
    {"email": "The CSV export is slightly misaligned in Firefox. Not urgent.", "expected": "Low"},
    {"email": "Payment processing is failing for enterprise clients. Revenue impact.", "expected": "High"},
    {"email": "Could you update the team page with my new photo when you get a chance?", "expected": "Low"},
]

def evaluate_prompt(prompt_template: str, cases: list[dict]) -> tuple[float, list[dict]]:
    """Score a prompt template on test cases. Returns (accuracy, results)."""
    results = []
    for case in cases:
        full_prompt = prompt_template + f"\n\nEmail: \"{case['email']}\"\nUrgency:"
        output = chat([{"role": "user", "content": full_prompt}], temperature=0.0).strip()
        correct = case["expected"].lower() in output.lower()
        results.append({"email": case["email"][:40], "expected": case["expected"],
                        "got": output[:20], "correct": correct})
    accuracy = sum(r["correct"] for r in results) / len(results)
    return accuracy, results


# Round 0: Weak initial prompt
current_prompt = "Classify the urgency of this email. Reply: High, Medium, or Low."
acc, results = evaluate_prompt(current_prompt, test_cases)
print(f"Round 0 prompt accuracy: {acc*100:.0f}%")
rprint(results)

# Round 1: Ask the model to improve the prompt
failures = [r for r in results if not r["correct"]]
optimize_prompt = f"""\
You are a prompt engineer. I have a prompt for classifying email urgency:

CURRENT PROMPT:
\"\"\"
{current_prompt}
\"\"\"

TEST RESULTS:
Accuracy: {acc*100:.0f}%
Failures: {failures}

TASK: Rewrite the prompt to fix these failures. 
Add clear criteria for High/Medium/Low (1-2 sentences each).
Keep it under 120 words. Output only the improved prompt, nothing else.
"""

improved_prompt = chat([{"role": "user", "content": optimize_prompt}], temperature=0.3)
show(improved_prompt, "Round 1 — Meta-Generated Improved Prompt", "blue")

# Test improved prompt
acc2, results2 = evaluate_prompt(improved_prompt, test_cases)
print(f"\nRound 1 prompt accuracy: {acc2*100:.0f}% (was {acc*100:.0f}%)")
rprint(results2)

Round 0 prompt accuracy: 80%


[
    {'email': "Server is down, 500 errors, users can't ", 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Can we schedule a call next week to disc', 'expected': 'Low', 'got': 'Medium', 'correct': False},
    {'email': 'The CSV export is slightly misaligned in', 'expected': 'Low', 'got': 'Low', 'correct': True},
    {'email': 'Payment processing is failing for enterp', 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Could you update the team page with my n', 'expected': 'Low', 'got': 'Low', 'correct': True}
]

╭─────────────────────────────────── Round 1 — Meta-Generated Improved Prompt ────────────────────────────────────╮
│ Classify the urgency of the email and reply with only "High", "Medium", or "Low".                               │
│                                                                                                                 │
│ High: Requires immediate action, deadline ≤ 24 hours, or addresses a critical problem.                          │
│ Medium: Important but can be handled within 2‑5 days; not a crisis but needs timely response.                   │
│ Low: Purely informational, routine, or can be addressed after a week without impact.                            │
│                                                                                                                 │
│ Provide the single word label.                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Round 1 prompt accuracy: 80% (was 80%)


[
    {'email': "Server is down, 500 errors, users can't ", 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Can we schedule a call next week to disc', 'expected': 'Low', 'got': 'Medium', 'correct': False},
    {'email': 'The CSV export is slightly misaligned in', 'expected': 'Low', 'got': 'Low', 'correct': True},
    {'email': 'Payment processing is failing for enterp', 'expected': 'High', 'got': 'High', 'correct': True},
    {'email': 'Could you update the team page with my n', 'expected': 'Low', 'got': 'Low', 'correct': True}
]

In [16]:
# ── EXAMPLE 8e · Code Self-Review Loop — generate, test, fix ─────────────────
# The model generates code, then reviews it as a critic,
# then produces a final corrected version.

coding_task = """\
Write a Python function `find_duplicates(lst)` that:
- Takes a list of any comparable items
- Returns a list of items that appear MORE than once (no duplicates in the output list)
- Preserves the order of first occurrence in the original list
- Handles empty lists gracefully
"""

# Step 1: Generate initial code
rprint(Rule("[bold]Step 1 — Generate Initial Code[/]"))
initial_code = chat([
    {"role": "user", "content": f"Write the function described below. Only output code, no explanation.\n\n{coding_task}"}
], temperature=0.5)
show(initial_code, "Initial Code (Generated)", "yellow")

# Step 2: Self-critique the code
rprint(Rule("[bold]Step 2 — Self-Critique[/]"))
critique_prompt = f"""\
Review this Python function for correctness, edge cases, and code quality.

Task it should solve:
{coding_task}

Code to review:
{initial_code}

Critique using these lenses:
1. CORRECTNESS — Does it produce the right output? Trace through 2-3 examples mentally.
2. EDGE CASES — What about: empty list, all duplicates, single element, mixed types?
3. TIME COMPLEXITY — What is O(n)? Is there a more efficient approach?
4. CODE QUALITY — Type hints? Docstring? Readable variable names?

Be specific about any bugs found.
"""
critique = chat([{"role": "user", "content": critique_prompt}], temperature=0.1)
show(critique, "Self-Critique", "red")

# Step 3: Revise based on critique
rprint(Rule("[bold]Step 3 — Revised Final Code[/]"))
revise_prompt = f"""\
Based on this critique, write the final corrected version of the function.
Address every issue mentioned. Include a docstring and type hints.
Output only the final code.

Critique:
{critique}
"""
final_code = chat([{"role": "user", "content": revise_prompt}], temperature=0.2)
show(final_code, "Final Corrected Code (After Self-Refinement)", "green")

# Actually execute and test the final code
print("\n🧪 Executing final code to verify...")
try:
    # Extract code from markdown fences if present
    code_only = re.sub(r'```python\n|```', '', final_code).strip()
    exec(code_only, globals())
    
    test_cases_code = [
        ([], []),
        ([1, 2, 3], []),
        ([1, 2, 2, 3, 3, 3], [2, 3]),
        (["a", "b", "a", "c", "b"], ["a", "b"]),
        ([1], []),
    ]
    all_pass = True
    for inp, expected in test_cases_code:
        result = find_duplicates(inp)  # type: ignore
        passed = result == expected
        status = "✓" if passed else "✗"
        print(f"  {status} find_duplicates({inp}) = {result} (expected {expected})")
        if not passed:
            all_pass = False
    print(f"\n{'✅ All tests passed!' if all_pass else '❌ Some tests failed.'}")
except Exception as e:
    print(f"❌ Execution error: {e}")

───────────────────────────────────────── Step 1 — Generate Initial Code ──────────────────────────────────────────

╭─────────────────────────────────────────── Initial Code (Generated) ────────────────────────────────────────────╮
│ ```python                                                                                                       │
│ def find_duplicates(lst):                                                                                       │
│     """                                                                                                         │
│     Return a list of items that appear more than once in `lst`,                                                 │
│     preserving the order of their first occurrence.                                                             │
│     """                                                                                                         │
│     seen = set()                                                                                                │
│     duplicates = []                                                                                             │
│     added = set()  # to avoid adding the same duplicate twice                                                   │
│                                                                                                                 │
│     for item in lst:                                                                                            │
│         if item in seen:                                                                                        │
│             if item not in added:                                                                               │
│                 duplicates.append(item)                                                                         │
│                 added.add(item)                                                                                 │
│         else:                                                                                                   │
│             seen.add(item)                                                                                      │
│                                                                                                                 │
│     return duplicates                                                                                           │
│ ```                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────────── Step 2 — Self-Critique ──────────────────────────────────────────────

╭───────────────────────────────────────────────── Self-Critique ─────────────────────────────────────────────────╮
│ ## Review of `find_duplicates`                                                                                  │
│                                                                                                                 │
│ ```python                                                                                                       │
│ def find_duplicates(lst):                                                                                       │
│     """                                                                                                         │
│     Return a list of items that appear more than once in `lst`,                                                 │
│     preserving the order of their first occurrence.                                                             │
│     """                                                                                                         │
│     seen = set()                                                                                                │
│     duplicates = []                                                                                             │
│     added = set()  # to avoid adding the same duplicate twice                                                   │
│                                                                                                                 │
│     for item in lst:                                                                                            │
│         if item in seen:                                                                                        │
│             if item not in added:                                                                               │
│                 duplicates.append(item)                                                                         │
│                 added.add(item)                                                                                 │
│         else:                                                                                                   │
│             seen.add(item)                                                                                      │
│                                                                                                                 │
│     return duplicates                                                                                           │
│ ```                                                                                                             │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. Correctness                                                                                              │
│                                                                                                                 │
│ | Input | Expected output | Actual output | Comments |                                                          │
│ |-------|----------------|---------------|----------|                                                           │
│ | `[1, 2, 3, 2, 4, 1]` | `[2, 1]` | ✅ | The first time a value is seen a second time it is appended, and later │
│ repeats are ignored. |                                                                                          │
│ | `['a', 'b', 'a', 'a', 'c']` | `['a']` | ✅ | Works for strings. |                                             │
│ | `[True, False, True, True]` | `[True]` | ✅ | Booleans are hashable, so they behave like integers. |          │
│ | `[]` | `[]` | ✅ | Returns an empty list. |             

─────────────────────────────────────────── Step 3 — Revised Final Code ───────────────────────────────────────────

╭───────────────────────────────── Final Corrected Code (After Self-Refinement) ──────────────────────────────────╮
│ ```python                                                                                                       │
│ from __future__ import annotations                                                                              │
│                                                                                                                 │
│ import math                                                                                                     │
│ from typing import Any, List, Sequence, TypeVar                                                                 │
│                                                                                                                 │
│ T = TypeVar("T")                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
│ def find_duplicates(seq: Sequence[T]) -> List[T]:                                                               │
│     """                                                                                                         │
│     Return a list of items that appear more than once in *seq*,                                                 │
│     preserving the order of their **first** occurrence.                                                         │
│                                                                                                                 │
│     The function works for any comparable items:                                                                │
│                                                                                                                 │
│     * If the items are hashable, it runs in O(n) time using sets.                                               │
│     * If an item is unhashable, it falls back to an O(n²) algorithm that                                        │
│       relies only on equality comparisons.                                                                      │
│                                                                                                                 │
│     ``float('nan')`` values are treated as duplicates of each other                                             │
│     even though ``nan != nan``.                                                                                 │
│     """                                                                                                         │
│     duplicates: List[T] = []                                                                                    │
│                                                                                                                 │
│     # ------------------------------------------------------------------                                        │
│     # Fast path – all elements are hashable                                                                     │
│     # ------------------------------------------------------------------                                        │
│     try:                                                                                                        │
│         seen: set[Any] = set()                                                                                  │
│         reported: set[Any] = set()                                                                              │
│         nan_seen = False                                                                                        │
│         nan_reported = False                                                                                    │
│                                                       


🧪 Executing final code to verify...
  ✓ find_duplicates([]) = [] (expected [])
  ✓ find_duplicates([1, 2, 3]) = [] (expected [])
  ✓ find_duplicates([1, 2, 2, 3, 3, 3]) = [2, 3] (expected [2, 3])
  ✓ find_duplicates(['a', 'b', 'a', 'c', 'b']) = ['a', 'b'] (expected ['a', 'b'])
  ✓ find_duplicates([1]) = [] (expected [])

✅ All tests passed!


### 🧠 Student Exercise 8
Build a **3-round self-refinement loop** for the following task:

Task: Generate a 5-slide presentation outline for **"Introduction to Large Language Models"** for a university audience.

Critique criteria to use:
1. Does every slide have a clear, specific purpose?
2. Is the progression logical (foundational → advanced)?
3. Does it include at least one interactive/demo slide?
4. Is the content appropriate for a university audience (not too basic, not too advanced)?

Print the outline after each round and note which criteria scores improved.

---
## Summary — Notebook 3 (and Full Course)

| Concept | Key Takeaway |
|---------|-------------|
| **Prompt Chaining** | Break complex tasks into single-purpose steps. Output of A → Input of B. Map-Reduce for bulk processing. Gates for routing. |
| **Meta-Prompting** | Let the LLM write and optimize its own prompts. Evaluate against test cases. Iterate. |
| **Self-Refinement** | Generate → Critique → Revise. 10–25% quality gain per loop. Use self-consistency voting for factual tasks. |

---

## Complete Course Summary

| # | Topic | Notebook | Key Technique |
|---|-------|----------|---------------|
| 1 | Anatomy of a Prompt | NB 1 | 4 components: instruction, context, input, output format |
| 2 | Zero/One/Few-Shot | NB 1 | 3 examples → 94% format compliance |
| 3 | System Prompt Design | NB 1 | Hierarchical: persona → tone → 3–5 rules |
| 4 | Chain-of-Thought | NB 2 | `<reasoning>` tags → +34% accuracy |
| 5 | Structured Output | NB 2 | Pydantic parse() / JSON mode / XML tags |
| 6 | Robustness Testing | NB 2 | Synonym tests + adversarial inputs + positive constraints |
| 7 | Prompt Chaining | NB 3 | Sequential / Map-Reduce / Parallel / Gated |
| 8 | Meta-Prompting | NB 3 | Generate → Critique → Revise loop |

---
*Good prompt engineering is **empirical** — always measure, always test, always iterate.*